# Flashcard Generator

In [1]:
import os
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List, Optional
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

## Pydantic Schema

In [2]:
class Flashcard(BaseModel):
    front: str = Field(description="The question, prompt, or term.")
    back: str = Field(description="The answer, explanation, or definition.")
    concept_tag: str = Field(description="Sub-topic tag (e.g., 'Cell Organelles', 'Genetics').")

class FlashcardCollection(BaseModel):
    title: str = Field(description="A descriptive title for this study deck.")
    cards: List[Flashcard] = Field(description="List of extracted flashcards.")

## Flashcard Generation Function

In [3]:
def generate_cards(
    extracted_elements: List, 
    num_cards: Union[int, Literal["Auto"]] = "Auto", 
    flashcard_mode: Literal["Terms & Definitions", "Practice Test", "Fill-in-the-Blank", "Auto"] = "Auto"
) -> Optional[FlashcardCollection]:
    """
    Takes a list of extracted text/images, adds system instructions, and forces 
    Gemini 3.1 Flash Lite to return a validated FlashcardCollection.
    """
    
    # System Instructions
    instructions = f"""
    You are an expert study assistant. Analyze the source material provided below, and generate a deck of flashcards.
    
    CRITICAL SETTINGS:
    1. Card Count Rule:
       - If num_cards is an integer, generate exactly that number of cards (currently: {num_cards}).
       - If num_cards is "Auto": analyze the depth of the material and determine the optimal number of cards.

    2. Flashcard Mode: {flashcard_mode}. 
       - If 'Terms & Definitions', focus on vocabulary and clear definitions.
       - If 'Practice Test', format the fronts of the cards like formal exam questions.
       - If 'Fill-in-the-Blank', create sentences with a missing key word/phrase represented by "___" on the front, and the missing word on the back.
       - If 'Auto', analyze the content and dynamically select the single most appropriate mode, or combine styles where it fits best.
"""
    
    # Combine prompt
    full_payload = [instructions] + extracted_elements
    
    try:
        print(f"Sending {len(extracted_elements)} elements to Gemini...")
        
        # Call Gemini API
        response = client.models.generate_content(
            model='gemini-3.1-flash-lite',
            contents=full_payload,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=FlashcardCollection,
                temperature=0.1
            )
        )
        
        # Return Python Object
        return response.parsed
        
    except Exception as e:
        print(f"Error during generation: {e}")
        return None

## Testing

In [4]:
from notes_extractor import process_document

### Extract Notes

In [5]:
pdf_elements = process_document("sample_files/sample.pdf")

### Generate Flashcards

In [6]:
deck = generate_cards(pdf_elements, num_cards="Auto", flashcard_mode="Auto")

Sending 4 elements to Gemini...


In [7]:
if deck:
    print("\n" + "="*50)
    print(f"{deck.title}")
    print("="*50)
    
    for idx, card in enumerate(deck.cards):
        print(f"\n--- Card {idx + 1} | Tag: [{card.concept_tag}] ---")
        print(f"Q: {card.front}")
        print(f"A: {card.back}")
else:
    print("Failed to generate the deck.")


The Incredible World of Cells

--- Card 1 | Tag: [Introduction to Cells] ---
Q: What is the basic building block of all living things?
A: A cell.

--- Card 2 | Tag: [Cell Theory] ---
Q: What are the three main rules of the Cell Theory?
A: 1. All living things are made of one or more cells. 2. The cell is the basic unit of structure and function in living things. 3. All cells come from pre-existing cells.

--- Card 3 | Tag: [Cell Types] ---
Q: What is the primary difference between prokaryotic and eukaryotic cells?
A: Eukaryotic cells have a nucleus that holds their DNA, while prokaryotic cells do not have a nucleus and their DNA floats freely.

--- Card 4 | Tag: [Organelles] ---
Q: Which organelle acts as the 'control center' of the cell and contains DNA?
A: The nucleus.

--- Card 5 | Tag: [Organelles] ---
Q: What is the function of the mitochondria in a cell?
A: They act as the power plants of the cell, turning sugar into usable energy through cellular respiration.

--- Card 6 | Tag: